# 阶段 0 · 第一张净值曲线

**标的**：华商优势行业A（000390.OF）——5 只持仓里成立最早（合同生效 2013-12-11），近 3 年数据完整。

**数据**：周度净值（每周最后一个交易日，日期列为周日标签），2023-09-24 → 2026-09-20 共 160 周。
来源 Wind（pick-funds 插件），取数与口径细节见 `data/README.md`；两个异常点已用同花顺独立源交叉验证。

**本 notebook 的四个目标**（阶段 0 的核心学习内容）：
1. 单位净值 vs 累计净值——基金世界里"价格"的两个口径
2. 数据会骗人——先画出来、再交叉验证
3. 涨跌看"倍数"，不看"点数"
4. 量化的本质：判断 → 规则 → 回测 → 证伪


## 概念 1：单位净值与累计净值

- **单位净值** = 基金总资产 ÷ 总份额。它是你申赎时**唯一的成交价格**：今天申购，就按成交规则那天的单位净值算你买到多少份。
- 基金**分红**时，钱分给了持有人，基金资产变少 → 单位净值等额下降。这个下降不是亏损。
- **累计净值** = 单位净值 + 每份历史累计分红。用途是还原"如果从成立起从没分过红"的净值走势。
- 两个可直接从数据验证的推论：
  - **累计净值 − 单位净值 = 每份累计分红**；这个差值的**跳变** = 一次分红事件
  - 无分红的周，两者**绝对变动相同**（累计=单位+常数）；涨幅**百分比**必然不同，因为分母不同


In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# macOS 中文字体：matplotlib 默认字体没有中文字形，不设置中文会渲染成方块
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang SC", "Hiragino Sans GB"]
plt.rcParams["axes.unicode_minus"] = False  # 负号正常显示

# 兼容从 notebooks/ 或项目根目录两种启动方式
csv = Path("../data/000390_weekly_nav.csv")
if not csv.exists():
    csv = Path("data/000390_weekly_nav.csv")
BASE = csv.resolve().parent.parent
FIG_DIR = BASE / "figures"
FIG_DIR.mkdir(exist_ok=True)

df = pd.read_csv(csv, parse_dates=["date"]).rename(
    columns={"unit_nav": "单位净值", "cum_nav": "累计净值"}
)
df.tail(3)

,date,单位净值,累计净值
154,2026-09-06,2.382,5.460
155,2026-09-13,2.606,5.684
156,2026-09-20,2.732,5.810


In [4]:
# 量化第一习惯：先验证数据，再相信数据
assert df["date"].is_monotonic_increasing, "日期必须升序"
assert df["date"].is_unique, "日期不能重复"
assert df.notna().all().all(), "不能有缺失值"

gap = df["累计净值"] - df["单位净值"]  # 每份累计分红（概念1的推论）

# 净值披露精度 3 位小数 → 差值有 ±0.002 的舍入抖动，容差要留足；
# 真正的分红（≥0.05 元/份）远大于抖动，用 0.005 做分红识别阈值
assert (gap.diff().dropna() >= -0.002).all(), "差值大幅下降 = 数据自相矛盾（分红只会让差值增大）"

print(f"区间 {df['date'].iloc[0].date()} → {df['date'].iloc[-1].date()}，共 {len(df)} 周")
print("差值(累计-单位)出现过的取值:", sorted(gap.round(2).unique()))
print("→ 差值从 2.33 一路跳到 3.08，每次跳变就是一次分红，稍后用图印证")
print(f"识别到分红事件: {(gap.diff() > 0.005).sum()} 次")

区间 2023-09-24 → 2026-09-20，共 157 周
差值(累计-单位)出现过的取值: [np.float64(2.33), np.float64(2.38), np.float64(2.61), np.float64(2.62), np.float64(3.08)]
→ 差值从 2.33 一路跳到 3.08，每次跳变就是一次分红，稍后用图印证
识别到分红事件: 3 次


上一步的体检不只是走形式：`差值只增不减` 这条断言，利用的正是概念 1 的数学关系——**数据一旦违反它，说明某个净值抄错了**。这个断言在本数据的制作过程中真实抓到过一处手抄错误（2024-12-01 的单位净值），把金融关系变成可机器验证的断言，这就是量化思维的最小样本。

## 概念 2：数据会骗人——先画出来，再交叉验证

这份原始数据里有两处一眼可疑的点：

- **2026-06-21 那周**：差值跳升 0.463（分红 0.463 元/份），但单位净值不降反升——除息 0.463 还能涨，隐含当周涨幅约 +16%
- **2026-08-09 那周**：单位净值一周 **+18.1%**，对一只灵活配置型基金极其反常

处理方式不是删除，而是**交叉验证**：换一个独立数据源（同花顺）核对同样的日期——两点完全一致。结论：这是基金 2026 年的真实波动（上半年约 +89%，随后 5 周 -38%，又一周反弹 +18%），不是数据错误。

> 教训：异常值是"理解的入口"，不是"删除的理由"。删除前先验证，验证不了再考虑丢弃并记录。


In [2]:
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(11, 8.5), sharex=True, gridspec_kw={"height_ratios": [3, 2]}
)

# ── 上图：两个口径的绝对净值 ──
ax1.plot(df["date"], df["单位净值"], lw=1.8, label="单位净值（你申赎的成交价）")
ax1.plot(df["date"], df["累计净值"], lw=1.8, alpha=0.85, label="累计净值（单位净值 + 每份历史分红）")

# 分红周 = 差值跳变的周（概念1推论的直接应用，阈值避开舍入抖动）
# 标注放在绘图区上方的外侧（x 用数据坐标、y 用坐标区比例），保证不压任何曲线
import matplotlib.transforms as mtransforms
trans = mtransforms.blended_transform_factory(ax1.transData, ax1.transAxes)
for idx, row in df.loc[gap.diff() > 0.005].iterrows():
    amt = gap.diff().loc[idx]
    ax1.axvline(row["date"], color="grey", ls=":", lw=1.2)
    ax1.text(row["date"], 1.02, f"分红{amt:+.3f}元", transform=trans,
             ha="center", fontsize=7.5, color="dimgrey")

ax1.set_title("华商优势行业A（000390）周度净值 2023-09 → 2026-09", fontsize=13, pad=16)
ax1.set_ylabel("净值（元）")
ax1.legend(loc="upper left")

# ── 下图：起点归一（看倍数）+ 最大回撤 ──
nav = df["单位净值"]
ax2.plot(df["date"], nav / nav.iloc[0], lw=1.8, label="单位净值（起点=1）")
ax2.plot(df["date"], df["累计净值"] / df["累计净值"].iloc[0], lw=1.5, alpha=0.8,
         label="累计净值（起点=1）")
ax2.axhline(1, color="grey", lw=0.8)

# 最大回撤：从历史最高点到其后最低点的跌幅（单位净值口径粗算，阶段 1 讲精确口径）
dd = nav / nav.cummax() - 1
trough = dd.idxmin()
peak = nav.loc[:trough].idxmax()
ax2.axvspan(df.loc[peak, "date"], df.loc[trough, "date"], color="red", alpha=0.12)
# 说明文字放在左上空白区（2023→2025 两年两条曲线都贴着 1，上方完全空），不压任何数据
ax2.text(pd.Timestamp("2024-06-01"), 2.45,
         f"红色阴影：最大回撤 {dd.loc[trough]:.1%}\n"
         f"（{df.loc[peak, 'date'].date()} → {df.loc[trough, 'date'].date()}，单位净值口径）",
         fontsize=9, color="red", va="top")
ax2.set_ylabel("相对起点倍数")
ax2.legend(loc="upper left")

fig.tight_layout()
fig.savefig(FIG_DIR / "000390_nav_3y.png", dpi=150, bbox_inches="tight")
print("图已保存:", FIG_DIR / "000390_nav_3y.png")
fig

NameError: name 'plt' is not defined

## 概念 3：涨跌看"倍数"，不看"点数"

上图（绝对净值）适合看事件：分红虚线、净值跳水与反弹一目了然。但**比较涨跌必须用下图（起点=1）**——因为绝对点数没有共同起点，"从 1 元涨到 2 元"和"从 3 元涨到 4 元"都是涨 1 元，含金量完全不同。

不过，即便是下图的两条线，**也都不是你的真实收益**：单位净值漏掉了"分红再投资"的复利，累计净值把分红简单加回、不做复利。看下面的数字感受口径差异有多大——正确答案（复权净值）是阶段 1 的第一课。


In [ ]:
r_unit = df["单位净值"].iloc[-1] / df["单位净值"].iloc[0] - 1
r_cum = df["累计净值"].iloc[-1] / df["累计净值"].iloc[0] - 1

summary = pd.Series(
    {
        "单位净值口径 3 年涨幅": f"{r_unit:+.1%}",   # 漏分红再投 → 低估
        "累计净值口径 3 年涨幅": f"{r_cum:+.1%}",   # 分红加回不复利 → 也不对
        "两个口径差": f"{abs(r_unit - r_cum):.1%} 个百分点",
        "最大回撤（单位净值）": f"{dd.min():.1%}",
        "分红次数": int((gap.diff() > 0.005).sum()),
        "真实收益": "需复权净值，阶段 1 解决",
    },
    name="000390 华商优势行业A",
)
summary.to_frame()

,000390 华商优势行业A
单位净值口径 3 年涨幅,+158.2%
累计净值口径 3 年涨幅,+71.5%
两个口径差,86.7% 个百分点
最大回撤（单位净值）,-37.6%
分红次数,3
真实收益,需复权净值，阶段 1 解决


## 概念 4：量化的本质

**主观交易**：判断 → 下单。判断对不对，靠记忆和感觉复盘，容易自我欺骗。
**量化**：判断 → **写成规则** → 规则在历史上**回放**（回测）→ 看证据决定信不信。

区别不是谁更聪明，是量化**可证伪**：规则写死了，历史收益算出来多少就是多少，赖不掉。

个人量化最常见的四种死法（本项目每一步都要对着检查）：

| 死法 | 一句话版本 | 本项目的护栏 |
|---|---|---|
| 过拟合 | 参数调到历史最漂亮 = 背下了历史考卷的答案 | 样本内/外分离（阶段 2） |
| 忽略成本 | 申赎费吃掉全部超额（持有<7天赎回费 1.5%） | 回测强制计成本（阶段 2） |
| 样本太短 | 3 年数据撑不起复杂策略 | 策略保持极简 |
| 运气当能力 | 测 20 个策略总有一个好看 | 记录测过的策略数量（阶段 4） |

还有一条本项目特有的第一性基准：**任何策略先和"等权买入持有 + 月度再平衡"比，跑不赢就没有存在意义。**

## 数据异常日志（诚实原则）

| 日期（周） | 现象 | 核验 | 结论 |
|---|---|---|---|
| 2026-06-21 | 差值跳升 0.463（分红），单位净值当周仍上涨 | 同花顺一致 | 真实：分红除息 + 当周大涨 |
| 2026-08-09 | 单位净值单周 +18.1%（无分红） | 同花顺一致 | 真实：暴跌后极限反弹周 |

背景：该基金 2026 上半年约 +89%，随后 5 周 -38%。**你在 70:30 的风险偏好下拿着它**——阶段 1 会量化这段经历的回撤与波动。

## 下一步（阶段 1 预告）

- 复权净值：投资者真实收益的正确口径
- 收益率（日/累计/年化，自算 `(1+R)^(1/n)-1`）、年化波动率、最大回撤、夏普/卡玛的正式计算
- 产出 `fund_metrics.py`：基金代码 + 时间段 → 净值曲线 + 回撤曲线 + 指标表

**读书作业**：《打开量化投资的黑箱》（Rishi Narang）概念部分。
